# Chunking and embeddings playground

Read a workbook, cut it into chunks, embed each chunk with Titan Text Embeddings v2, then retrieve the closest chunks
for a query. Every step is deliberately simple so it is easy to swap out.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").is_dir():  # walk up to the repo root wherever the kernel started
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")  # AWS_PROFILE locally; leave it unset where the machine has its own role

WORKBOOK = ROOT / "data" / "document_library" / "documents" / "raw" / "Location of 386 Pokémon in Gen 3.xlsx"
SHEETS = ["Gen 3 Location Sheet"]  # or "all" for every sheet
CHUNKING = "row"                   # "row" or "row_group"
ROWS_PER_CHUNK = 10                # used by "row_group"

## 1. Read the sheets

In [ ]:
sheets = pd.read_excel(WORKBOOK, sheet_name=None, dtype=str, keep_default_na=False)
print(list(sheets))
selected = sheets if SHEETS == "all" else {name: sheets[name] for name in SHEETS}
{name: frame.shape for name, frame in selected.items()}

## 2. Chunk

- `row`: one chunk per row, written as `column: value` lines so each chunk carries its own headers. Empty cells are skipped.
- `row_group`: `ROWS_PER_CHUNK` consecutive rows per chunk, as CSV with the header once. Fewer, denser chunks.

Every chunk starts with its sheet name and row range so a hit can be traced back to the workbook.

In [ ]:
def row_chunks(name, frame):
    for i, row in frame.iterrows():
        body = "\n".join(f"{col}: {val}" for col, val in row.items() if str(val).strip())
        if body:
            yield f"Sheet: {name} (row {i + 2})\n{body}"  # +2: Excel is 1-based and row 1 is the header

def row_group_chunks(name, frame):
    frame = frame.loc[frame.apply(lambda r: any(str(v).strip() for v in r), axis=1)]  # drop blank rows
    for start in range(0, len(frame), ROWS_PER_CHUNK):
        group = frame.iloc[start:start + ROWS_PER_CHUNK]
        rows = f"rows {group.index[0] + 2}-{group.index[-1] + 2}"
        yield f"Sheet: {name} ({rows})\n{group.to_csv(index=False).strip()}"

make_chunks = {"row": row_chunks, "row_group": row_group_chunks}[CHUNKING]
chunks = [chunk for name, frame in selected.items() for chunk in make_chunks(name, frame)]
print(f"{len(chunks)} chunks, longest {max(map(len, chunks)):,} characters")
too_long = [c for c in chunks if len(c) > 30_000]  # Titan v2 takes at most 8,192 tokens, roughly 30k characters
if too_long:
    print(f"Warning: {len(too_long)} chunks may exceed Titan's input limit; lower ROWS_PER_CHUNK")
print()
print(chunks[0])

## 3. Embed

Same call as `amazon.titan-embed-text-v2:0` on the VDI. Titan v2 accepts up to 8,192 tokens per input; `normalize` makes cosine similarity a plain dot product.

In [ ]:
bedrock = boto3.Session().client("bedrock-runtime", region_name="us-east-1")

def embed(text):
    response = bedrock.invoke_model(modelId="amazon.titan-embed-text-v2:0",
                                    body=json.dumps({"inputText": text, "dimensions": 1024, "normalize": True}))
    return json.loads(response["body"].read())["embedding"]

with ThreadPoolExecutor(max_workers=8) as pool:
    vectors = np.array(list(pool.map(embed, chunks)))
vectors.shape

## 4. Retrieve

In [ ]:
def retrieve(query, k=5):
    scores = vectors @ np.array(embed(query))
    best = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]

QUERY = "Where can I find Ralts?"  # change me
for score, chunk in retrieve(QUERY):
    print(f"--- {score:.3f}\n{chunk[:500]}\n")

## 5. Your experiments